# AI Lab - Practical Question No. 21
## Missionaries and Cannibals Problem using Depth-First Search (DFS)

- **Syllabus Area**: Uninformed Search Techniques
- **Difficulty**: 🔴 Expert
- **Practical Type**: State-Space Search
- **Core AI Concept**: Constraint Checking in State Generation, Implicit Graph Traversal, Loop Avoidance
- **Python Concepts Tested**: Complex condition evaluation, tuple representation, recursive & iterative DFS
- **Expected Algorithm / Technique**: Depth-First Search (DFS) with cycle detection
- **Recommended Python Libraries**: None (Pure Python Standard Library)
- **Complexity Expectation**: Time/Space bounded by small state space ($|S| \le 32$, $|S_{\text{safe}}| = 20$, $|S_{\text{reachable}}| = 16$)
- **Constraints / Conditions**: DFS must avoid loops; at no time can cannibals outnumber missionaries on either bank
- **Input Requirements**: None (Standard 3M, 3C parameters)
- **Expected Output**: Trace of states leading to `(0, 0, 'Right')`
- **Sample Input**: Execute search
- **Sample Output**: `Step 1: 2C cross right... [Full path trace]`
- **Optional Challenge**: Animate the text output with delays
- **University Practical Importance**: Very High

---

### Problem Statement
Three missionaries (M) and three cannibals (C) must cross a river using a boat that holds up to two people (capacity = 2). The boat cannot cross without at least one person rowing (minimum passengers = 1). At no time, on either riverbank, can cannibals outnumber missionaries present on that bank, unless there are zero missionaries on that bank (if M == 0, cannibals have no missionaries to eat).

### Task Requirements
1. **State Representation**: `(M_left, C_left, Boat_pos)` where $0 \le M_l \le 3$, $0 \le C_l \le 3$, and $\text{Boat} \in \{\text{'Left'}, \text{'Right'}\}$.
2. **Generate 5 Valid Moves**: `1M`, `2M`, `1C`, `2C`, `1M1C`.
3. **Validate States**: Ensure physical availability of passengers and enforce safety constraints on both banks.
4. **Output Sequence of Boat Trips**: From initial state `(3, 3, 'Left')` to goal state `(0, 0, 'Right')`.
5. **Cycle Detection**: Ensure DFS avoids infinite loops and cycles.
6. **Optional Challenge**: Animate the text output with delays.

### 1. Theoretical Foundations: State-Space Representation & Constraint Validation

#### State Representation
A state is represented as a 3-tuple: `(M_left, C_left, Boat_pos)`.
- **Left Bank**: $M_l$ missionaries, $C_l$ cannibals.
- **Right Bank**: Derived by mass conservation: $M_r = 3 - M_l$, $C_r = 3 - C_l$.
- **Initial State**: $s_0 = (3, 3, \text{'Left'})$
- **Goal State**: $s_g = (0, 0, \text{'Right'})$

#### The 5 Canonical Moves
The boat can hold at least 1 and at most 2 passengers:
1. `1M`   : (1, 0) - 1 Missionary crosses
2. `2M`   : (2, 0) - 2 Missionaries cross
3. `1C`   : (0, 1) - 1 Cannibal crosses
4. `2C`   : (0, 2) - 2 Cannibals cross
5. `1M1C` : (1, 1) - 1 Missionary and 1 Cannibal cross

#### Safety Constraints
A state `(M_left, C_left, Boat_pos)` is safe **if and only if**:
1. $0 \le M_l \le 3$ and $0 \le C_l \le 3$
2. If $M_l > 0$, then $M_l \ge C_l$ (Left bank missionaries are safe)
3. If $M_r > 0$, then $M_r \ge C_r$ (Right bank missionaries are safe, where $M_r = 3 - M_l$, $C_r = 3 - C_l$)

In [ ]:
import sys
from pathlib import Path

# Ensure project root is accessible
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.missionaries_cannibals_dfs import (
    BOAT_LEFT,
    BOAT_RIGHT,
    CANONICAL_5_MOVES,
    DEFAULT_TOTAL_MISSIONARIES,
    DEFAULT_TOTAL_CANNIBALS,
    INITIAL_STATE,
    GOAL_STATE,
    MissionariesCannibalsSolver,
    analyze_state_space,
    animate_solution,
    format_solution_trace,
    get_missionaries_cannibals_viva_reference,
    is_valid_state,
    render_ascii_river,
    solve_missionaries_cannibals_dfs,
    validate_state_detailed,
)

print("Missionaries and Cannibals module loaded successfully.")
print(f"Initial State : {INITIAL_STATE}")
print(f"Goal State    : {GOAL_STATE}")
print(f"The 5 Moves   : {[m.label for m in CANONICAL_5_MOVES]}")

### 2. Solving the Problem with Depth-First Search (DFS)

We now execute Depth-First Search with loop avoidance. Because river-crossing transitions are reversible, tracking visited states along the search branch is necessary to avoid infinite loops.

In [ ]:
solver = MissionariesCannibalsSolver()
result = solver.solve_dfs(method="iterative")

print("=== STANDARD DFS SOLUTION TRACE ===\n")
print(format_solution_trace(result, format_type="standard"))

print(f"\nSearch Statistics:")
print(f"  • Total Boat Trips : {result.total_trips}")
print(f"  • States Explored  : {result.states_explored}")
print(f"  • States Generated : {result.states_generated}")
print(f"  • Cycles Avoided   : {result.pruned_by_cycle}")
print(f"  • Execution Time   : {result.execution_time_ms:.3f} ms")

### 3. Step-by-Step Tabular Trace and ASCII Visualizer

Below is the complete state transition table showing the distribution of missionaries and cannibals across both banks and in transit over the river.

In [ ]:
print(format_solution_trace(result, format_type="table"))

In [ ]:
print(format_solution_trace(result, format_type="arrows"))

### 4. Constraint Validation: Generation vs. Testing

- **Validation during Generation (Forward Checking)**: Preconditions and safety rules are evaluated before generating child nodes. Invalid actions are pruned before entering the frontier.
- **Validation during Testing (Generate-and-Test)**: All 5 actions are generated blindly, and constraint checking is performed afterwards.

In [ ]:
res_gen = solver.solve_dfs(validation_mode="generation")
res_test = solver.solve_dfs(validation_mode="testing")

print(f"+------------------------------+--------------------+--------------------+")
print(f"| Metric                       | During Generation  | During Testing     |")
print(f"+------------------------------+--------------------+--------------------+")
print(f"| Pruned by Capacity Bounds    | {res_gen.pruned_by_capacity:<18} | {res_test.pruned_by_capacity:<18} |")
print(f"| Pruned by Safety Constraint  | {res_gen.pruned_by_safety:<18} | {res_test.pruned_by_safety:<18} |")
print(f"| Pruned by Cycle Detection    | {res_gen.pruned_by_cycle:<18} | {res_test.pruned_by_cycle:<18} |")
print(f"| Execution Time (ms)          | {res_gen.execution_time_ms:<18.3f} | {res_test.execution_time_ms:<18.3f} |")
print(f"+------------------------------+--------------------+--------------------+")

### 5. State Space Combinatorial Analysis

We exhaustively enumerate all states in the problem domain to understand:
1. Theoretical unconstrained states: $(3+1) \times (3+1) \times 2 = 32$
2. Valid (safe) states satisfying cannibal constraints: exactly 20 states
3. Reachable safe states from `(3, 3, 'Left')`: exactly 16 states

In [ ]:
analysis = analyze_state_space()
print(f"Theoretical States : {analysis['total_theoretical_states']}")
print(f"Safe States        : {analysis['total_safe_states']} / 32")
print(f"Reachable States   : {analysis['total_reachable_safe_states']}")
print(f"Unreachable States : {analysis['total_unreachable_safe_states']}\n")

print("Reachable Safe States:")
for i, st in enumerate(analysis['reachable_safe_states'], 1):
    print(f"  [{i:02d}] {st}")

### 6. Algorithm Comparison: Why BFS Might Be Preferred Over DFS Here

In university examinations, a fundamental question is:
> *Why might Breadth-First Search (BFS) be preferred over Depth-First Search (DFS) for this problem?*

1. **Optimality**: BFS is mathematically guaranteed to find the shortest path (minimum boat trips = 11).
2. **Cycle Sensitivity**: DFS can get trapped in deep or infinite cycles on reversible state spaces unless cycle detection is implemented.
3. **Small State Space**: Memory is $\mathcal{O}(V) \le 32$, so BFS's memory overhead is negligible.

In [ ]:
cmp_data = solver.compare_dfs_vs_bfs()
dfs = cmp_data['dfs']
bfs = cmp_data['bfs']

print(f"+------------------------------+--------------------+--------------------+")
print(f"| Metric                       | DFS (LIFO Stack)   | BFS (FIFO Queue)   |")
print(f"+------------------------------+--------------------+--------------------+")
print(f"| Trips (Path Length)          | {dfs['trips']:<18} | {bfs['trips']:<18} |")
print(f"| Is Path Shortest (Optimal)?  | {str(dfs['optimal']):<18} | {str(bfs['optimal']):<18} |")
print(f"| States Explored              | {dfs['states_explored']:<18} | {bfs['states_explored']:<18} |")
print(f"| States Generated             | {dfs['states_generated']:<18} | {bfs['states_generated']:<18} |")
print(f"+------------------------------+--------------------+--------------------+")

### 7. Optional Challenge: Text Animation with Delays

The optional challenge asks to animate the river crossing text output with delays. Below, we demonstrate the animation routine.

In [ ]:
# Execute animated trace with small delay
animate_solution(result, delay=0.15)

### 8. Viva Voce Examination Reference Guide

Below are the authoritative answers to the Viva Voce topics specified in the syllabus.

In [ ]:
viva = get_missionaries_cannibals_viva_reference()
for topic, content in viva.items():
    print(f"=======================================================================")
    print(f"[{topic}]")
    print(f"Question: {content['question']}\n")
    print(f"Answer:")
    for line in content['answer'].split('\n'):
        if line.strip():
            print(f"  • {line.strip()}")
    print()